# Paper 1: test data and saved-model evaluation

Use this separate notebook after training. The original four notebooks and model definitions are unchanged. Official test labels are private. The final output is a server submission, unless you supply a genuinely unused labelled holdout. Read `README_TEST_EVALUATION.md` for six-class scoring and ACDC metadata requirements.


In [ ]:
from pathlib import Path
import subprocess

KIT = Path("/tmp/asera/NEW/paper1_test_evaluation_v1")
PYTHON = "/nas.dbms/asera/micromamba/envs/CUDA_12.8_env/bin/python"
# For YOLO11m or RT-DETR-L, use /tmp/asera/paper1_reviewer34_env/bin/python
TEST_ROOT = Path("/tmp/asera/paper1_original/TEST_DATASET")
RUNS = Path("/tmp/asera/paper1_original/test_evaluations")
METHOD = "lgf_original"
TRAIN_DATASET = "coco_nw"
SEED = 42
SELECTION = "best_map"
CHECKPOINT = Path("/absolute/path/to/verified_selected_checkpoint.pth")
ACDC_ARCHIVE = None  # Set to Path("/actual/path/rgb_anon_trainvaltest.zip") for ACDC.

def run(script, *arguments):
    subprocess.run([PYTHON, str(KIT / script), *map(str, arguments)], cwd=KIT, check=True)


## COCO download and preparation

The first cell downloads roughly 6 GB of test images. Preparation checks the existing clear/weather training and validation splits. Run once, then reuse the generated manifest.


In [ ]:
run("test_data.py", "download", "--dataset", "coco", "--test-root", TEST_ROOT)


In [ ]:
run("test_data.py", "prepare", "--dataset", "coco_nw", "--test-root", TEST_ROOT)


## Synthetic weather, when required

This creates a fixed mixed-weather version using the original weather functions. Existing training images remain unchanged.


In [ ]:
run("test_data.py", "weather", "--condition", "mixed", "--weather-seed", 42, "--test-root", TEST_ROOT)


## ACDC, when required

Download the RGB archive through your ACDC account, or reuse your previous full archive. Set `ACDC_ARCHIVE` above. Only adverse test images are extracted. Skip these cells for a COCO-only run.


In [ ]:
if ACDC_ARCHIVE is None:
    raise ValueError("Set ACDC_ARCHIVE to your official downloaded RGB archive before running this cell.")
run("test_data.py", "download", "--dataset", "acdc", "--archive", ACDC_ARCHIVE, "--test-root", TEST_ROOT)
run("test_data.py", "prepare", "--dataset", "acdc", "--test-root", TEST_ROOT)


## Export the selected checkpoint

Set the checkpoint, training dataset, method, seed, and verified validation selection rule above. Use the same rule for all compared methods. ACDC exports need official ID metadata for submission packaging.


In [ ]:
manifest_names = {"coco_nw": "coco_nw_test.json", "coco_weather": "coco_weather_mixed_test.json", "acdc": "acdc_test.json"}
MANIFEST = TEST_ROOT / manifest_names[TRAIN_DATASET]
OUTPUT = RUNS / TRAIN_DATASET / METHOD / f"seed_{SEED}"
if not CHECKPOINT.is_file():
    raise FileNotFoundError("Set CHECKPOINT to the selected saved model.")
EXPORT_ARGS = ["export", "--method", METHOD, "--train-dataset", TRAIN_DATASET, "--seed", SEED, "--level", "C3",
               "--checkpoint", CHECKPOINT, "--selection", SELECTION, "--manifest", MANIFEST,
               "--device", "0", "--batch-size", "4", "--output", OUTPUT]
run("evaluate_test.py", *EXPORT_ARGS)


## Restart after interruption

Run this cell only to continue the same export. Completed batches are reused.


In [ ]:
run("evaluate_test.py", *EXPORT_ARGS, "--resume")


## Read export status

`metrics: null` means AP has not been calculated. COCO produces `submission.zip`. ACDC requires the official image/category mapping described in the README. Do not treat an 80-class COCO or eight-class ACDC server aggregate as six-class Paper 1 AP.


In [ ]:
import json
report = json.loads((OUTPUT / "export_report.json").read_text())
print(json.dumps({key: report.get(key) for key in ["metric_status", "metrics", "submission_zip", "server_mapping_verified", "prediction_file"]}, indent=2))
